# RTMDet-Ins-tiny for E-waste Battery Instance Segmentation

This notebook is a separate Google Colab notebook for training **RTMDet-Ins-tiny** on the single-class e-waste battery instance segmentation dataset.

It intentionally uses a separate **Python 3.10 micromamba environment** because the current Colab Python 3.12 stack can break OpenMMLab/MIM/MMCV installation. The notebook commands run RTMDet training through that Python 3.10 environment while the normal Colab kernel is only used for setup and file preparation.

Expected dataset layout in Google Drive:

```text
E-waste Battery Extraction CV/
  final_dataset/
    images/
      train/
      val/
      test/
    labels/
      train/
      val/
      test/
```

The YOLO segmentation labels are converted into COCO instance segmentation JSON files before training.

## 0. Runtime notes

Recommended Colab runtime:

```text
Runtime type: Python 3
Hardware accelerator: GPU
Preferred GPU: A100
```

This notebook does **not** retrain your YOLO or Detectron2 models. It only trains and evaluates RTMDet-Ins-tiny.

In [1]:
# ============================================================
# 0. GPU check
# ============================================================

!nvidia-smi

import os
import json
import time
import shutil
import subprocess
import re
from pathlib import Path

import numpy as np
import pandas as pd

print("Notebook kernel Python is only used for setup.")

Sat May  2 06:22:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Google Drive and define paths

Change `PROJECT_ROOT` only if your dataset is stored somewhere else.

In [2]:
# ============================================================
# 1. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# 1.1 Define global paths and settings
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/E-waste Battery Extraction CV")
DATASET_ROOT = PROJECT_ROOT / "final_dataset"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_NAMES = ["battery"]
NUM_CLASSES = 1

# Output folder for this RTMDet-only notebook
RTMDET_ROOT = PROJECT_ROOT / "03_rtmdet_models"
RTMDET_RUN_NAME = "model_07_rtmdet_ins_tiny"
RTMDET_WORK_DIR = RTMDET_ROOT / RTMDET_RUN_NAME
RTMDET_WORK_DIR.mkdir(parents=True, exist_ok=True)

COCO_DIR = PROJECT_ROOT / "rtmdet_coco_annotations"
COCO_DIR.mkdir(parents=True, exist_ok=True)

# Training settings
IMG_SIZE = 640
RTMDET_BATCH_SIZE = 16       # If CUDA OOM occurs, reduce to 8.
RTMDET_NUM_WORKERS = 4
RTMDET_EPOCHS = 100
RTMDET_VAL_INTERVAL = 5
RTMDET_SAVE_INTERVAL = 5
RTMDET_LR = 0.00025

# Paths for micromamba environment
MAMBA_EXE = Path("/content/micromamba-bin/micromamba")
MAMBA_ROOT_PREFIX = Path("/content/micromamba")
ENV_NAME = "rtmdet310"
ENV_PYTHON = f"{MAMBA_ROOT_PREFIX}/envs/{ENV_NAME}/bin/python"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("RTMDET_WORK_DIR:", RTMDET_WORK_DIR)
print("COCO_DIR:", COCO_DIR)
print("RTMDET_BATCH_SIZE:", RTMDET_BATCH_SIZE)
print("RTMDET_EPOCHS:", RTMDET_EPOCHS)

assert DATASET_ROOT.exists(), f"Missing DATASET_ROOT: {DATASET_ROOT}"

PROJECT_ROOT: /content/drive/MyDrive/E-waste Battery Extraction CV
DATASET_ROOT: /content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset
RTMDET_WORK_DIR: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny
COCO_DIR: /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations
RTMDET_BATCH_SIZE: 16
RTMDET_EPOCHS: 100


## 2. Dataset checks

This verifies that the YOLO segmentation layout exists.

In [4]:
# ============================================================
# 2. Dataset checks
# ============================================================

def count_split(split):
    img_dir = DATASET_ROOT / "images" / split
    lab_dir = DATASET_ROOT / "labels" / split
    imgs = [p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXTS] if img_dir.exists() else []
    labs = list(lab_dir.glob("*.txt")) if lab_dir.exists() else []
    return len(imgs), len(labs)

for split in ["train", "val", "test"]:
    n_img, n_lab = count_split(split)
    print(f"{split:5s}: images={n_img}, labels={n_lab}")

for split in ["train", "val", "test"]:
    assert (DATASET_ROOT / "images" / split).exists(), f"Missing images/{split}"
    assert (DATASET_ROOT / "labels" / split).exists(), f"Missing labels/{split}"

train: images=1235, labels=1235
val  : images=97, labels=97
test : images=97, labels=97


## 3. Convert YOLO segmentation labels to COCO JSON

RTMDet-Ins expects COCO instance segmentation annotations. This cell converts your YOLO polygon labels into COCO JSON files for train, validation, test, and an optional selected test subset.

The selected test subset follows your rule:

```text
For each original test image:
1. include the original image;
2. if it has at least four augmentations, include _aug_1, _aug_2, _aug_3;
3. otherwise include all available augmentations.
```

In [ ]:
# ============================================================
# 3. Convert YOLO segmentation labels to COCO format
# ============================================================

import cv2
import json
import re
from pathlib import Path
from tqdm import tqdm
import numpy as np

AUG_RE = re.compile(r"^(?P<base>.+)_aug_(?P<idx>\d+)$")

def is_augmented_stem(stem):
    return AUG_RE.match(stem) is not None

def base_from_stem(stem):
    m = AUG_RE.match(stem)
    return m.group("base") if m else stem

def aug_index_from_stem(stem):
    m = AUG_RE.match(stem)
    return int(m.group("idx")) if m else None

def get_image_files(split):
    img_dir = DATASET_ROOT / "images" / split
    return sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS])

def selected_test_image_paths():
    all_test_images = get_image_files("test")
    groups = {}
    originals = {}

    for img_path in all_test_images:
        stem = img_path.stem
        base = base_from_stem(stem)
        groups.setdefault(base, []).append(img_path)
        if not is_augmented_stem(stem):
            originals[base] = img_path

    selected = []

    for base, original_path in sorted(originals.items()):
        selected.append(original_path)

        aug_paths = [
            p for p in groups.get(base, [])
            if is_augmented_stem(p.stem)
        ]
        aug_paths = sorted(aug_paths, key=lambda p: aug_index_from_stem(p.stem))

        if len(aug_paths) >= 4:
            preferred = []
            for idx in [1, 2, 3]:
                matches = [p for p in aug_paths if aug_index_from_stem(p.stem) == idx]
                preferred.extend(matches)
            selected_aug = preferred if len(preferred) == 3 else aug_paths[:3]
        else:
            selected_aug = aug_paths

        selected.extend(selected_aug)

    return selected

def yolo_split_to_coco(split, out_json, selected_paths=None):
    if selected_paths is None:
        img_paths = get_image_files(split)
    else:
        img_paths = sorted(selected_paths)

    lab_dir = DATASET_ROOT / "labels" / split

    images = []
    annotations = []
    ann_id = 1
    img_id = 1

    for img_path in tqdm(img_paths, desc=f"COCO convert {split}"):
        img = cv2.imread(str(img_path))
        if img is None:
            print("Skipping unreadable image:", img_path)
            continue

        h, w = img.shape[:2]

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": w,
            "height": h
        })

        label_path = lab_dir / f"{img_path.stem}.txt"

        if label_path.exists() and label_path.read_text().strip():
            lines = label_path.read_text().strip().splitlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) < 7:
                    continue

                cls_id = int(float(parts[0]))
                coords = list(map(float, parts[1:]))

                if len(coords) % 2 != 0:
                    coords = coords[:-1]

                poly = []
                xs, ys = [], []

                for i in range(0, len(coords), 2):
                    x = float(coords[i] * w)
                    y = float(coords[i + 1] * h)
                    x = max(0.0, min(float(w - 1), x))
                    y = max(0.0, min(float(h - 1), y))
                    poly.extend([x, y])
                    xs.append(x)
                    ys.append(y)

                if len(xs) < 3:
                    continue

                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                box_w = x_max - x_min
                box_h = y_max - y_min

                if box_w <= 1 or box_h <= 1:
                    continue

                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                area = float(abs(cv2.contourArea(pts)))

                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": 1,
                    "segmentation": [poly],
                    "bbox": [float(x_min), float(y_min), float(box_w), float(box_h)],
                    "area": area,
                    "iscrowd": 0
                })

                ann_id += 1

        img_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": 1, "name": "battery", "supercategory": "object"}
        ]
    }

    out_json = Path(out_json)
    out_json.parent.mkdir(parents=True, exist_ok=True)

    with open(out_json, "w") as f:
        json.dump(coco, f)

    print(f"Saved {out_json}")
    print(f"Images: {len(images)}, annotations: {len(annotations)}")
    return out_json

TRAIN_COCO = yolo_split_to_coco("train", COCO_DIR / "train.json")
VAL_COCO = yolo_split_to_coco("val", COCO_DIR / "val.json")
TEST_COCO = yolo_split_to_coco("test", COCO_DIR / "test.json")

SELECTED_TEST_PATHS = selected_test_image_paths()
SELECTED_TEST_COCO = yolo_split_to_coco(
    "test",
    COCO_DIR / "test_selected_original_plus_aug.json",
    selected_paths=SELECTED_TEST_PATHS
)

print("Selected test images:", len(SELECTED_TEST_PATHS))

COCO convert train: 100%|██████████| 1235/1235 [23:19<00:00,  1.13s/it]


Saved /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/train.json
Images: 1235, annotations: 1237


COCO convert val: 100%|██████████| 97/97 [04:32<00:00,  2.81s/it]


Saved /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/val.json
Images: 97, annotations: 97


COCO convert test: 100%|██████████| 97/97 [04:27<00:00,  2.76s/it]


Saved /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/test.json
Images: 97, annotations: 97


COCO convert test: 100%|██████████| 48/48 [00:01<00:00, 26.86it/s]


Saved /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/test_selected_original_plus_aug.json
Images: 48, annotations: 48
Selected test images: 48


## 4. Install a clean Python 3.10 micromamba environment

This avoids the Python 3.12 `pkgutil.ImpImporter` / OpenMMLab installation issue. All MMDetection commands later are executed inside this environment.

In [22]:
%%bash
rm -rf /content/micromamba/envs/rtmdet310
rm -rf /content/mmdetection

In [23]:
# ============================================================
# 4. Install micromamba and create Python 3.10 environment
# ============================================================

import os
from pathlib import Path

MAMBA_EXE = Path("/content/micromamba-bin/micromamba")
MAMBA_ROOT_PREFIX = Path("/content/micromamba")
ENV_NAME = "rtmdet310"

if not MAMBA_EXE.exists():
    !mkdir -p /content/micromamba-bin
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/micromamba-bin --strip-components=1 bin/micromamba
else:
    print("micromamba already exists:", MAMBA_EXE)

# Create the environment if it does not already exist.
env_python_path = MAMBA_ROOT_PREFIX / "envs" / ENV_NAME / "bin" / "python"

if not env_python_path.exists():
    !MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}" "{MAMBA_EXE}" create -y -n "{ENV_NAME}" python=3.10 pip
else:
    print("Environment already exists:", env_python_path)

!MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}" "{MAMBA_EXE}" run -n "{ENV_NAME}" python --version
!MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}" "{MAMBA_EXE}" run -n "{ENV_NAME}" pip --version

micromamba already exists: /content/micromamba-bin/micromamba
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)

Resolving Environment                                                                     ✔ Done (0.6 sec)

Transaction

  Prefix: /content/micromamba/envs/rtmdet310

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel           Size
───

## 5. Install PyTorch, MMCV, MMEngine, and MMDetection in the Python 3.10 environment

This notebook avoids `openmim` and installs the MMCV wheel directly for **Torch 2.1 + CUDA 12.1**, which is usually more stable than using the current Colab Python 3.12 / Torch 2.10 stack.

In [24]:
%%bash
set -e

export MAMBA_ROOT_PREFIX="/content/micromamba"
MAMBA="/content/micromamba-bin/micromamba"

$MAMBA run -n rtmdet310 python -m pip install -U pip setuptools wheel packaging

$MAMBA run -n rtmdet310 python -m pip install \
"numpy<2" pandas matplotlib tqdm opencv-python pycocotools scipy pyyaml

$MAMBA run -n rtmdet310 python -m pip install \
torch==2.1.2 torchvision==0.16.2 \
--index-url https://download.pytorch.org/whl/cu121

$MAMBA run -n rtmdet310 python -m pip install \
"mmcv==2.1.0" \
-f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

$MAMBA run -n rtmdet310 python -m pip install \
"mmengine>=0.10.3" terminaltables rich addict yapf shapely prettytable seaborn

MPLBACKEND=Agg $MAMBA run -n rtmdet310 python - <<'PY'
import sys
import torch
import mmcv
import mmengine
import numpy

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("MMCV:", mmcv.__version__)
print("MMEngine:", mmengine.__version__)
print("NumPy:", numpy.__version__)
PY

  Using cached pip-26.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached pycocotools-2.0.11-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (1.3 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-ma

In [7]:
# ============================================================
# Verify imports inside the micromamba Python 3.10 environment
# ============================================================

CHECK_SCRIPT = Path("/content/check_rtmdet_env.py")

CHECK_SCRIPT.write_text("""
import sys
import torch
import mmcv
import mmengine
import numpy

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("MMCV:", mmcv.__version__)
print("MMEngine:", mmengine.__version__)
print("NumPy:", numpy.__version__)
""")

!MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}" "{MAMBA_EXE}" run -n "{ENV_NAME}" python "{CHECK_SCRIPT}"

Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
Torch: 2.1.2+cu121
Torch CUDA: 12.1
CUDA available: True
MMCV: 2.1.0
MMEngine: 0.10.7
NumPy: 1.26.4


## 6. Clone MMDetection and install it into the Python 3.10 environment

This provides the RTMDet-Ins config files and `tools/train.py` / `tools/test.py`.

In [26]:
%%bash
set -e

export MAMBA_ROOT_PREFIX="/content/micromamba"
MAMBA="/content/micromamba-bin/micromamba"

$MAMBA run -n rtmdet310 python -m pip install \
"pip<25" \
"setuptools==69.5.1" \
"wheel==0.43.0" \
"packaging==24.2"

$MAMBA run -n rtmdet310 python - <<'PY'
import setuptools
import pkg_resources
import packaging

print("setuptools:", setuptools.__version__)
print("pkg_resources: OK")
print("packaging:", packaging.__version__)
PY

  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 22.5 MB/s  0:00:00
Using cached packaging-24.2-py3-none-any.whl (65 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.1 MB/s  0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1
  Attempting uninstall: pip
    Found existing installation: pip 26.1
    Uninstalling pip-26.1:
      Successfully uninstalled pip-26.1
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2

setuptools: 69.5.1
pkg_resources: OK
packaging: 24.2


<stdin>:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html


In [27]:
%%bash
set -e

export MAMBA_ROOT_PREFIX="/content/micromamba"
export MPLBACKEND="Agg"

MAMBA="/content/micromamba-bin/micromamba"
MMDET_DIR="/content/mmdetection"

if [ ! -d "$MMDET_DIR" ]; then
    git clone -q --branch v3.3.0 https://github.com/open-mmlab/mmdetection.git "$MMDET_DIR"
else
    echo "MMDetection repo already exists: $MMDET_DIR"
fi

cd "$MMDET_DIR"

$MAMBA run -n rtmdet310 python -m pip install -v -e . --no-build-isolation

PYTHONPATH="$MMDET_DIR:$PYTHONPATH" MPLBACKEND=Agg \
$MAMBA run -n rtmdet310 python - <<'PY'
from pathlib import Path
import mmdet
from mmdet.utils import register_all_modules

register_all_modules(init_default_scope=True)

cfg = Path("/content/mmdetection/configs/rtmdet/rtmdet-ins_tiny_8xb32-300e_coco.py")

print("MMDetection:", mmdet.__version__)
print("MMDetection path:", mmdet.__file__)
print("RTMDet-Ins-tiny config exists:", cfg.exists())
print("Config:", cfg)

assert cfg.exists(), f"Missing RTMDet config: {cfg}"
PY

MMDetection repo already exists: /content/mmdetection
Using pip 24.3.1 from /content/micromamba/envs/rtmdet310/lib/python3.10/site-packages/pip (python 3.10)
Obtaining file:///content/mmdetection
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for mmdet
MMDetection: 3.3.0
MMDetection path: /content/mmdetection/mmdet/__init__.py
RTMDet-Ins-tiny config exists: True
Config: /content/mmdetection/configs/rtmdet/rtmdet-ins_tiny_8xb32-300e_coco.py


  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info
  writing /tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/requires.txt
  writing top-level names to /tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/SOURCES.txt'
  reading manifest template 'MANIFEST.in'
  adding license file 'LICENSE'
  writing manifest file '/tmp/pip-pip-egg-info-hohn3yo7/mmdet.egg-info/SOURCES.txt'
  DEPRECATION: Legacy editable install of mmdet==3.3.0 from file:///content/mmdetection (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or

## 7. Copy COCO annotations into the dataset folder

The custom config uses `DATASET_ROOT` as `data_root`, so the annotation JSON files are copied into `final_dataset/mmdet_annotations/`.

In [ ]:
# ============================================================
# 7. Copy COCO annotations into final_dataset/mmdet_annotations
# ============================================================

MMDET_ANN_DIR = DATASET_ROOT / "mmdet_annotations"
MMDET_ANN_DIR.mkdir(parents=True, exist_ok=True)

annotation_map = {
    "train.json": TRAIN_COCO,
    "val.json": VAL_COCO,
    "test.json": TEST_COCO,
    "test_selected_original_plus_aug.json": SELECTED_TEST_COCO,
}

for dst_name, src_path in annotation_map.items():
    dst_path = MMDET_ANN_DIR / dst_name
    shutil.copy2(src_path, dst_path)
    print(f"Copied {src_path} -> {dst_path}")

print("Annotation folder:", MMDET_ANN_DIR)
print(list(MMDET_ANN_DIR.glob("*.json")))

Copied /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/train.json -> /content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset/mmdet_annotations/train.json
Copied /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/val.json -> /content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset/mmdet_annotations/val.json
Copied /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/test.json -> /content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset/mmdet_annotations/test.json
Copied /content/drive/MyDrive/E-waste Battery Extraction CV/rtmdet_coco_annotations/test_selected_original_plus_aug.json -> /content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset/mmdet_annotations/test_selected_original_plus_aug.json
Annotation folder: /content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset/mmdet_annotations
[PosixPath('/content/drive/MyDrive/E-waste Battery Extraction CV/final_dat

## 8. Create a custom RTMDet-Ins-tiny config

This config adapts the COCO RTMDet-Ins-tiny config to your one-class battery dataset.

In [28]:
# ============================================================
# 8. Create custom RTMDet-Ins-tiny config
# ============================================================

RTMDET_CUSTOM_CONFIG = RTMDET_WORK_DIR / "rtmdet_ins_tiny_battery.py"

cfg_text = f'''
_base_ = r"{MMDET_DIR}/configs/rtmdet/rtmdet-ins_tiny_8xb32-300e_coco.py"

metainfo = dict(
    classes=("battery",),
    palette=[(0, 255, 0)]
)

data_root = r"{str(DATASET_ROOT)}/"

model = dict(
    bbox_head=dict(num_classes=1)
)

train_dataloader = dict(
    batch_size={RTMDET_BATCH_SIZE},
    num_workers={RTMDET_NUM_WORKERS},
    persistent_workers=True,
    dataset=dict(
        type="CocoDataset",
        data_root=data_root,
        metainfo=metainfo,
        ann_file="mmdet_annotations/train.json",
        data_prefix=dict(img="images/train/"),
        filter_cfg=dict(filter_empty_gt=True, min_size=32)
    )
)

val_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=False,
    drop_last=False,
    dataset=dict(
        type="CocoDataset",
        data_root=data_root,
        metainfo=metainfo,
        ann_file="mmdet_annotations/val.json",
        data_prefix=dict(img="images/val/"),
        test_mode=True
    )
)

test_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=False,
    drop_last=False,
    dataset=dict(
        type="CocoDataset",
        data_root=data_root,
        metainfo=metainfo,
        ann_file="mmdet_annotations/test_selected_original_plus_aug.json",
        data_prefix=dict(img="images/test/"),
        test_mode=True
    )
)

val_evaluator = dict(
    type="CocoMetric",
    ann_file=data_root + "mmdet_annotations/val.json",
    metric=["bbox", "segm"],
    format_only=False
)

test_evaluator = dict(
    type="CocoMetric",
    ann_file=data_root + "mmdet_annotations/test_selected_original_plus_aug.json",
    metric=["bbox", "segm"],
    format_only=False
)

train_cfg = dict(
    type="EpochBasedTrainLoop",
    max_epochs={RTMDET_EPOCHS},
    val_interval={RTMDET_VAL_INTERVAL}
)

val_cfg = dict(type="ValLoop")
test_cfg = dict(type="TestLoop")

optim_wrapper = dict(
    optimizer=dict(lr={RTMDET_LR})
)

param_scheduler = [
    dict(
        type="LinearLR",
        start_factor=1.0e-5,
        by_epoch=False,
        begin=0,
        end=1000
    ),
    dict(
        type="CosineAnnealingLR",
        eta_min=1.0e-6,
        begin=0,
        end={RTMDET_EPOCHS},
        T_max={RTMDET_EPOCHS},
        by_epoch=True,
        convert_to_iter_based=True
    )
]

default_hooks = dict(
    checkpoint=dict(
        type="CheckpointHook",
        interval={RTMDET_SAVE_INTERVAL},
        max_keep_ckpts=3,
        save_best="coco/segm_mAP",
        rule="greater"
    ),
    logger=dict(type="LoggerHook", interval=20)
)

work_dir = r"{str(RTMDET_WORK_DIR)}"

env_cfg = dict(cudnn_benchmark=True)
'''

RTMDET_CUSTOM_CONFIG.write_text(cfg_text)

print("Created custom config:", RTMDET_CUSTOM_CONFIG)
print(RTMDET_CUSTOM_CONFIG.read_text()[:4000])

Created custom config: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/rtmdet_ins_tiny_battery.py

_base_ = r"/content/mmdetection/configs/rtmdet/rtmdet-ins_tiny_8xb32-300e_coco.py"

metainfo = dict(
    classes=("battery",),
    palette=[(0, 255, 0)]
)

data_root = r"/content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset/"

model = dict(
    bbox_head=dict(num_classes=1)
)

train_dataloader = dict(
    batch_size=16,
    num_workers=4,
    persistent_workers=True,
    dataset=dict(
        type="CocoDataset",
        data_root=data_root,
        metainfo=metainfo,
        ann_file="mmdet_annotations/train.json",
        data_prefix=dict(img="images/train/"),
        filter_cfg=dict(filter_empty_gt=True, min_size=32)
    )
)

val_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=False,
    drop_last=False,
    dataset=dict(
        type="CocoDataset",
        data_root=data_root,
        metainfo=me

## 9. Sanity-check the custom config

If this cell fails, fix the config before starting training.

In [29]:
# ============================================================
# 9. Print/verify custom config
# ============================================================

import os
import subprocess

resolved_config = RTMDET_WORK_DIR / "config_resolved.py"

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT_PREFIX)
env["PYTHONPATH"] = f"{MMDET_DIR}:{env.get('PYTHONPATH', '')}"
env["MPLBACKEND"] = "Agg"

cmd = [
    str(MAMBA_EXE),
    "run",
    "-n",
    ENV_NAME,
    "python",
    str(MMDET_DIR / "tools/misc/print_config.py"),
    str(RTMDET_CUSTOM_CONFIG),
    "--save-path",
    str(resolved_config),
]

ret = subprocess.run(
    cmd,
    cwd=str(MMDET_DIR),
    env=env,
    text=True,
    capture_output=True,
)

print("Return code:", ret.returncode)
print("STDOUT:")
print(ret.stdout)
print("STDERR:")
print(ret.stderr)

if ret.returncode != 0:
    raise RuntimeError("Config check failed. Do not trust config_resolved.py yet.")

print("Resolved config saved to:", resolved_config)

Return code: 0
STDOUT:
Config:
auto_scale_lr = dict(base_batch_size=16, enable=False)
backend_args = None
base_lr = 0.004
checkpoint = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/cspnext_rsb_pretrain/cspnext-tiny_imagenet_600e.pth'
custom_hooks = [
    dict(
        ema_type='ExpMomentumEMA',
        momentum=0.0002,
        priority=49,
        type='EMAHook',
        update_buffers=True),
    dict(
        switch_epoch=280,
        switch_pipeline=[
            dict(backend_args=None, type='LoadImageFromFile'),
            dict(
                poly2mask=False,
                type='LoadAnnotations',
                with_bbox=True,
                with_mask=True),
            dict(
                keep_ratio=True,
                ratio_range=(
                    0.5,
                    2.0,
                ),
                scale=(
                    640,
                    640,
                ),
                type='RandomResize'),
            dict(
              

In [11]:
# ============================================================
# 9. Print/verify custom config
# ============================================================

resolved_config = RTMDET_WORK_DIR / "config_resolved.py"

!MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}" "{MAMBA_EXE}" run -n "{ENV_NAME}" python "{MMDET_DIR}/tools/misc/print_config.py" "{RTMDET_CUSTOM_CONFIG}" --save-path "{resolved_config}"

print("Resolved config saved to:", resolved_config)

Traceback (most recent call last):
  File "/content/mmdetection/tools/misc/print_config.py", line 7, in <module>
    from mmdet.utils import replace_cfg_vals, update_data_root
ModuleNotFoundError: No module named 'mmdet'
Resolved config saved to: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/config_resolved.py


## 10. Train RTMDet-Ins-tiny

This saves checkpoints and logs to Google Drive:

```text
03_rtmdet_models/model_07_rtmdet_ins_tiny/
```

If you get CUDA out-of-memory, reduce `RTMDET_BATCH_SIZE` to `8`, rerun the config cell, and rerun training.

In [30]:
# ============================================================
# Pre-training import check
# ============================================================

import os
import subprocess

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT_PREFIX)
env["PYTHONPATH"] = f"{MMDET_DIR}:{env.get('PYTHONPATH', '')}"
env["MPLBACKEND"] = "Agg"

cmd = [
    str(MAMBA_EXE),
    "run",
    "-n",
    ENV_NAME,
    "python",
    "-c",
    (
        "import torch, mmcv, mmengine, mmdet; "
        "print('torch', torch.__version__); "
        "print('cuda', torch.cuda.is_available()); "
        "print('mmcv', mmcv.__version__); "
        "print('mmengine', mmengine.__version__); "
        "print('mmdet', mmdet.__version__)"
    ),
]

ret = subprocess.run(
    cmd,
    cwd=str(MMDET_DIR),
    env=env,
    text=True,
    capture_output=True,
)

print("Return code:", ret.returncode)
print("STDOUT:")
print(ret.stdout)
print("STDERR:")
print(ret.stderr)

if ret.returncode != 0:
    raise RuntimeError("Pre-training import check failed. Do not train yet.")

Return code: 0
STDOUT:
torch 2.1.2+cu121
cuda True
mmcv 2.1.0
mmengine 0.10.7
mmdet 3.3.0

STDERR:



In [32]:
# ============================================================
# Train RTMDet-Ins Tiny with live Colab log + saved log file
# ============================================================

import os
import subprocess
from pathlib import Path

train_log = RTMDET_WORK_DIR / "train_rtmdet_ins_tiny_live.log"
RTMDET_WORK_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT_PREFIX)
env["PYTHONPATH"] = f"{MMDET_DIR}:{env.get('PYTHONPATH', '')}"
env["MPLBACKEND"] = "Agg"

cmd = [
    str(MAMBA_EXE),
    "run",
    "-n",
    ENV_NAME,
    "python",
    str(MMDET_DIR / "tools/train.py"),
    str(RTMDET_CUSTOM_CONFIG),
    "--work-dir",
    str(RTMDET_WORK_DIR),
]

print("Running command:")
print(" ".join(cmd))
print("\nLive training log will appear below.")
print("Full log will also be saved to:")
print(train_log)
print("=" * 80)

with open(train_log, "w", encoding="utf-8") as f:
    process = subprocess.Popen(
        cmd,
        cwd=str(MMDET_DIR),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")   # show live in Colab
        f.write(line)         # save to Drive log file
        f.flush()

    process.wait()

print("=" * 80)
print("Return code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError("RTMDet-Ins Tiny training failed. Check the live log above or the saved log file.")

print("Training finished successfully.")
print("Outputs saved to:", RTMDET_WORK_DIR)

Running command:
/content/micromamba-bin/micromamba run -n rtmdet310 python /content/mmdetection/tools/train.py /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/rtmdet_ins_tiny_battery.py --work-dir /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny

Live training log will appear below.
Full log will also be saved to:
/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/train_rtmdet_ins_tiny_live.log
05/02 07:08:04 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1996803422
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: gcc (Ubuntu 11.4.

## 11. Locate the best checkpoint

MMDetection should save a `best_coco_segm_mAP_*.pth` checkpoint. If it does not, this cell falls back to the latest epoch checkpoint.

In [33]:
# ============================================================
# 11. Locate best checkpoint
# ============================================================

import re
from pathlib import Path

def extract_epoch_num(path):
    m = re.search(r"epoch_(\d+)", path.name)
    return int(m.group(1)) if m else -1

best_ckpts = sorted(RTMDET_WORK_DIR.glob("best*.pth"))

if len(best_ckpts) > 0:
    RTMDET_BEST_CKPT = best_ckpts[-1]
    print("Using best checkpoint:", RTMDET_BEST_CKPT)
else:
    epoch_ckpts = sorted(RTMDET_WORK_DIR.glob("epoch_*.pth"), key=extract_epoch_num)
    assert len(epoch_ckpts) > 0, f"No checkpoints found in {RTMDET_WORK_DIR}"
    RTMDET_BEST_CKPT = epoch_ckpts[-1]
    print("No best checkpoint found. Using latest epoch:", RTMDET_BEST_CKPT)

print("Exists:", RTMDET_BEST_CKPT.exists())
print("Size MB:", RTMDET_BEST_CKPT.stat().st_size / 1024**2)

Using best checkpoint: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/best_coco_segm_mAP_epoch_90.pth
Exists: True
Size MB: 47.62919330596924


## 12. Evaluate RTMDet-Ins-tiny on the selected test subset

This uses the selected test subset annotation file generated earlier.

In [34]:
# ============================================================
# 12. Evaluate RTMDet-Ins-tiny on the selected test subset
# ============================================================

import subprocess
from pathlib import Path

TEST_LOG = RTMDET_WORK_DIR / "test_selected_stdout.txt"
EVAL_WORK_DIR = RTMDET_WORK_DIR / "eval_selected_test"

RTMDET_BEST_CKPT = Path(RTMDET_BEST_CKPT)
RTMDET_CUSTOM_CONFIG = Path(RTMDET_CUSTOM_CONFIG)

if not RTMDET_CUSTOM_CONFIG.exists():
    raise FileNotFoundError(f"Custom config not found: {RTMDET_CUSTOM_CONFIG}")

if not RTMDET_BEST_CKPT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {RTMDET_BEST_CKPT}")

EVAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = f'''
set -o pipefail

export MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}"
export PYTHONPATH="{MMDET_DIR}:$PYTHONPATH"
export MPLBACKEND="Agg"

"{MAMBA_EXE}" run -n "{ENV_NAME}" \
python "{MMDET_DIR}/tools/test.py" \
"{RTMDET_CUSTOM_CONFIG}" \
"{RTMDET_BEST_CKPT}" \
--work-dir "{EVAL_WORK_DIR}" \
2>&1 | tee "{TEST_LOG}"
'''

print("Test command:")
print(cmd)

ret = subprocess.run(
    cmd,
    shell=True,
    executable="/bin/bash",
    cwd=str(MMDET_DIR),
)

print("Test return code:", ret.returncode)
print("Test log:", TEST_LOG)
print("Eval work dir:", EVAL_WORK_DIR)

if ret.returncode != 0:
    raise RuntimeError("RTMDet-Ins Tiny evaluation failed. Check the log above.")
else:
    print("Evaluation finished successfully.")

Test command:

set -o pipefail

export MAMBA_ROOT_PREFIX="/content/micromamba"
export PYTHONPATH="/content/mmdetection:$PYTHONPATH"
export MPLBACKEND="Agg"

"/content/micromamba-bin/micromamba" run -n "rtmdet310" python "/content/mmdetection/tools/test.py" "/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/rtmdet_ins_tiny_battery.py" "/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/best_coco_segm_mAP_epoch_90.pth" --work-dir "/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/eval_selected_test" 2>&1 | tee "/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/test_selected_stdout.txt"

Test return code: 0
Test log: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/test_selected_stdout.txt
Eval work dir: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_m

## 13. Parse RTMDet COCO metrics

This extracts bbox/segm AP values from the MMDetection evaluation log when available.

In [36]:
# ============================================================
# 13. Parse RTMDet COCO metrics from test log
# ============================================================

import re
import json
import numpy as np
from pathlib import Path

def parse_mmdet_coco_stdout(text):
    result = {
        "bbox_mAP": np.nan,
        "bbox_mAP_50": np.nan,
        "bbox_mAP_75": np.nan,
        "segm_mAP": np.nan,
        "segm_mAP_50": np.nan,
        "segm_mAP_75": np.nan,
    }

    # Format: bbox_mAP_copypaste: 0.xxx 0.xxx 0.xxx ...
    for key_prefix, out_prefix in [
        ("bbox_mAP_copypaste", "bbox"),
        ("segm_mAP_copypaste", "segm"),
    ]:
        pattern = rf"{key_prefix}:\s*([-0-9.\s]+)"
        m = re.search(pattern, text)
        if m:
            nums = [float(x) for x in m.group(1).strip().split()]
            if len(nums) >= 3:
                result[f"{out_prefix}_mAP"] = nums[0]
                result[f"{out_prefix}_mAP_50"] = nums[1]
                result[f"{out_prefix}_mAP_75"] = nums[2]

    # Format: coco/segm_mAP: 0.xxx or 'coco/segm_mAP': 0.xxx
    for raw_key, out_key in [
        ("coco/bbox_mAP", "bbox_mAP"),
        ("coco/bbox_mAP_50", "bbox_mAP_50"),
        ("coco/bbox_mAP_75", "bbox_mAP_75"),
        ("coco/segm_mAP", "segm_mAP"),
        ("coco/segm_mAP_50", "segm_mAP_50"),
        ("coco/segm_mAP_75", "segm_mAP_75"),
    ]:
        pattern = rf"['\"]?{re.escape(raw_key)}['\"]?\s*[:=]\s*([-0-9.]+)"
        m = re.search(pattern, text)
        if m:
            result[out_key] = float(m.group(1))

    return result

if not TEST_LOG.exists():
    raise FileNotFoundError(f"Test log not found: {TEST_LOG}. Run the evaluation cell first.")

test_text = TEST_LOG.read_text(encoding="utf-8", errors="ignore")
rtmdet_coco_metrics = parse_mmdet_coco_stdout(test_text)

summary = {
    "Model": "RTMDet-Ins-tiny",
    "Checkpoint": str(RTMDET_BEST_CKPT),
    "Test_Log": str(TEST_LOG),
    "Selected_Test_COCO_Metrics": rtmdet_coco_metrics,
}

summary_path = RTMDET_WORK_DIR / "rtmdet_selected_test_metrics.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved summary to:", summary_path)

if np.isnan(rtmdet_coco_metrics["segm_mAP"]):
    print("\nWARNING: segm_mAP is NaN. This usually means the parser did not find COCO metrics in the test log.")
    print("Check whether the evaluation cell finished successfully and whether the log contains segm_mAP or segm_mAP_copypaste.")

{
  "Model": "RTMDet-Ins-tiny",
  "Checkpoint": "/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/best_coco_segm_mAP_epoch_90.pth",
  "Test_Log": "/content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/test_selected_stdout.txt",
  "Selected_Test_COCO_Metrics": {
    "bbox_mAP": 0.872,
    "bbox_mAP_50": 1.0,
    "bbox_mAP_75": 0.97,
    "segm_mAP": 0.357,
    "segm_mAP_50": 0.401,
    "segm_mAP_75": 0.389
  }
}
Saved summary to: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/rtmdet_selected_test_metrics.json


## 14. Count model parameters

This builds the RTMDet model and counts parameters from the custom config.

In [37]:
# ============================================================
# 14. Count RTMDet-Ins-tiny parameters
# ============================================================

import os
import subprocess
import json
from pathlib import Path

PARAM_SCRIPT = RTMDET_WORK_DIR / "count_params_rtmdet.py"
PARAM_JSON = RTMDET_WORK_DIR / "parameter_count.json"

if not RTMDET_CUSTOM_CONFIG.exists():
    raise FileNotFoundError(f"Custom config not found: {RTMDET_CUSTOM_CONFIG}")

if not RTMDET_BEST_CKPT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {RTMDET_BEST_CKPT}")

PARAM_SCRIPT.write_text(f'''
from mmengine.config import Config
from mmdet.registry import MODELS
from mmdet.utils import register_all_modules
from mmengine.runner import load_checkpoint
import json

register_all_modules(init_default_scope=True)

cfg = Config.fromfile(r"{str(RTMDET_CUSTOM_CONFIG)}")
model = MODELS.build(cfg.model)

# Loading the checkpoint is not strictly required for parameter counting,
# but it verifies that the checkpoint matches the model architecture.
load_checkpoint(model, r"{str(RTMDET_BEST_CKPT)}", map_location="cpu")

params_m = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

out = {{
    "parameters_m": round(params_m, 4),
    "trainable_parameters_m": round(trainable_params_m, 4)
}}

print(json.dumps(out, indent=2))

with open(r"{str(PARAM_JSON)}", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)
''')

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT_PREFIX)
env["PYTHONPATH"] = f"{MMDET_DIR}:{env.get('PYTHONPATH', '')}"
env["MPLBACKEND"] = "Agg"

cmd = [
    str(MAMBA_EXE),
    "run",
    "-n",
    ENV_NAME,
    "python",
    str(PARAM_SCRIPT),
]

ret = subprocess.run(
    cmd,
    cwd=str(MMDET_DIR),
    env=env,
    text=True,
    capture_output=True,
)

print("Return code:", ret.returncode)
print("STDOUT:")
print(ret.stdout)
print("STDERR:")
print(ret.stderr)

if ret.returncode != 0:
    raise RuntimeError("Parameter counting failed.")

print("Saved:", PARAM_JSON)

Return code: 0
STDOUT:
Loads checkpoint by local backend from path: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/best_coco_segm_mAP_epoch_90.pth
{
  "parameters_m": 5.6152,
  "trainable_parameters_m": 5.6152
}

STDERR:

Saved: /content/drive/MyDrive/E-waste Battery Extraction CV/03_rtmdet_models/model_07_rtmdet_ins_tiny/parameter_count.json


## 15. Optional: predict and save qualitative visualisation

This runs RTMDet inference on one selected test image and saves a visualised prediction to the model folder.

In [ ]:
# ============================================================
# 15. Qualitative example
# ============================================================

# Pick one selected test image.
QUAL_IMAGE = SELECTED_TEST_PATHS[0]
QUAL_OUT_DIR = RTMDET_WORK_DIR / "qualitative"
QUAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Qualitative image:", QUAL_IMAGE)
print("Output folder:", QUAL_OUT_DIR)

cmd = f'''
MAMBA_ROOT_PREFIX="{MAMBA_ROOT_PREFIX}" "{MAMBA_EXE}" run -n "{ENV_NAME}" python "{MMDET_DIR}/demo/image_demo.py" "{QUAL_IMAGE}" "{RTMDET_CUSTOM_CONFIG}" --weights "{RTMDET_BEST_CKPT}" --device cuda:0 --pred-score-thr 0.25 --out-dir "{QUAL_OUT_DIR}"
'''

print("Qualitative command:")
print(cmd)

ret = subprocess.run(cmd, shell=True, executable="/bin/bash")
print("Return code:", ret.returncode)

print("Generated files:")
for p in QUAL_OUT_DIR.rglob("*"):
    print(p)

## 16. Optional: export final artefact paths

Use these paths when you integrate RTMDet-Ins-tiny into your final results table.

In [ ]:
# ============================================================
# 16. Final paths
# ============================================================

print("RTMDet custom config:")
print(RTMDET_CUSTOM_CONFIG)

print("\nRTMDet best checkpoint:")
print(RTMDET_BEST_CKPT)

print("\nRTMDet work directory:")
print(RTMDET_WORK_DIR)

print("\nMetric summary:")
print(RTMDET_WORK_DIR / "rtmdet_selected_test_metrics.json")

print("\nParameter count:")
print(RTMDET_WORK_DIR / "parameter_count.json")